In [60]:
import json
import os
import base64
import krippendorff
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

In [12]:
ANN_1_FILES = [Path('aj_iaa_3.json'), Path('aj_main.json')]
ANN_2_FILES = [Path('ar_iaa_3.json'), Path('ar_main.json')]

In [50]:
NUM_ANNOTATORS = 2

In [5]:
with open(ANN_1_FILES[0], 'r') as fp:
    res = json.load(fp)

In [13]:
annotations_1 = []
annotations_2 = []
for res_file in ANN_1_FILES:
    with open(res_file, 'r') as fp:
        res = json.load(fp)
        annotations_1.append(res)

for res_file in ANN_2_FILES:
    with open(res_file, 'r') as fp:
        res = json.load(fp)
        annotations_2.append(res)

In [25]:
annotations_1[1][1]

{'id': 3792,
 'annotations': [{'id': 1608,
   'completed_by': 1,
   'result': [],
   'was_cancelled': True,
   'ground_truth': False,
   'created_at': '2026-05-19T06:24:19.045207Z',
   'updated_at': '2026-05-19T06:24:19.045220Z',
   'draft_created_at': None,
   'lead_time': 15.688,
   'prediction': {},
   'result_count': 0,
   'unique_id': '7a64b76e-c64a-474d-91de-7ba3593ab222',
   'import_id': None,
   'last_action': None,
   'bulk_created': False,
   'task': 3792,
   'project': 35,
   'updated_by': 1,
   'parent_prediction': None,
   'parent_annotation': None,
   'last_created_by': None}],
 'file_upload': 'c5804818-consolidated_edits.json',
 'drafts': [],
 'predictions': [],
 'data': {'id': '2409.14664',
  'flattened_file_path': '/Users/aadi.joshi/Desktop/scientific_fact_checking/tabular_scientific_fact_checking/code/latex_scraper/workspace/arxiv_sources/2409.14664/flat_main.tex',
  'pdf_file_path': '/data/local-files/?d=Desktop/scientific_fact_checking/tabular_scientific_fact_checki

In [28]:
len(annotations_1[1])

165

In [41]:
def get_choice_value(annotation):
    annotations = annotation['annotations']
    for ann in annotations:
        if ann['was_cancelled']:
            return None
        else:
            for res in ann['result']:
                if res['type'] == 'choices' and res['from_name'] == 'edit_correct':
                    choice = res['value']['choices'][0]
                    if choice == 'yes':
                        return 1
                    else:
                        return 0

In [45]:
#create meta_id for matching same task across annotators
matched_annotations = {}
for round_id in range(len(annotations_1)):
    for annotation in annotations_1[round_id]:
        paper_id = annotation['data']['id']
        error_id = annotation['data']['error_id']
        meta_id = f'{paper_id}___{error_id}___{round_id}'
        annotation_value = get_choice_value(annotation)
        matched_annotations[meta_id] = [annotation_value]
    
    for annotation in annotations_2[round_id]:
        paper_id = annotation['data']['id']
        error_id = annotation['data']['error_id']
        meta_id = f'{paper_id}___{error_id}___{round_id}'
        annotation_value = get_choice_value(annotation)
        matched_annotations[meta_id] = matched_annotations.get(meta_id, [None]) + [annotation_value]

In [56]:
final_annotations = {}
for k, v in matched_annotations.items():
    if len(v) < NUM_ANNOTATORS:
        continue

    # CHECK IF CORRECT FOR IAA COMPUTATION
    if None in v:
        continue

    final_annotations[k] = v

In [63]:
def print_iaa_scores(iaa_scores):
    for category, score in iaa_scores.items():
        category_str = category
        if category_str != 'overall':
            category_str = base64.urlsafe_b64decode(category).decode()
        #base64.urlsafe_b64encode(b"methodology_result_conflict").decode()
        print(f'{category_str:50} {score:5.3}')

In [90]:
def compute_iaa_score(annotations, iaa_metric='cohens_kappa'):
    annotator_results = {'overall': [[] for _ in range(NUM_ANNOTATORS)]}
    for meta_id, responses in annotations.items():
        manuscript_id, error_id, round_id = meta_id.split('___')

        for i, res in enumerate(responses):
            annotator_results['overall'][i].append(res)

            if error_id not in annotator_results:
                annotator_results[error_id] = [[] for _ in range(NUM_ANNOTATORS)]
            
            annotator_results[error_id][i].append(res)

    iaa_scores = {}
    for category, responses in annotator_results.items():
        if iaa_metric == 'cohens_kappa':
            iaa_scores[category] = cohen_kappa_score(*responses, labels=[0, 1])
        elif iaa_metric == 'gwets_ac1':
            #annoying, but copy this into R notebook
            #install.packages("irrCAC")
            #library(irrCAC)
            #then copy each output
            outp = "array1 <- c("
            for r in responses[0]:
                outp += str(r) + ','
            outp = outp[:-1] + ')\n'
            outp += "array2 <- c("
            for r in responses[1]:
                outp += str(r) + ','
            outp = outp[:-1] + ')\n'
            outp += "annotations <- data.frame(array1, array2)\n"
            if category != 'overall':
                outp += "print('" + base64.urlsafe_b64decode(category).decode() + "')\n"
            else:
                outp += "print('Overall')\n"
            outp += r"print(gwet.ac1.raw(annotations)$est$coeff.val)"

            print(outp)
        elif iaa_metric == 'percent_agreement':
            iaa_scores[category] = np.mean(np.array(responses[0]) == np.array(responses[1]))
        elif iaa_metric == 'krippendorff_alpha':
            iaa_scores[category] = krippendorff.alpha(*responses)
            # Implement Krippendorff's alpha if needed
    return iaa_scores

In [67]:
iaa_metric = 'percent_agreement'
iaa_scores = compute_iaa_score(final_annotations, iaa_metric=iaa_metric)
print(f"{'category':50} {iaa_metric:5}")
print_iaa_scores(iaa_scores)

category                                           percent_agreement
overall                                            0.895
methodology_result_conflict                          1.0
logically_invalid_interpretation                   0.667
semantic_misinterpretation                          0.75
suboptimal_methodology                               1.0
contradiction_with_established_knowledge             1.0


In [91]:
iaa_metric = 'gwets_ac1'
iaa_scores = compute_iaa_score(final_annotations, iaa_metric=iaa_metric)
#print(f"{'category':50} {iaa_metric:5}")
#print_iaa_scores(iaa_scores)

array1 <- c(1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1)
array2 <- c(1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1)
annotations <- data.frame(array1, array2)
print('Overall')
print(gwet.ac1.raw(annotations)$est$coeff.val)
array1 <- c(1,1,1,1)
array2 <- c(1,1,1,1)
annotations <- data.frame(array1, array2)
print('methodology_result_conflict')
print(gwet.ac1.raw(annotations)$est$coeff.val)
array1 <- c(1,0,1)
array2 <- c(1,1,1)
annotations <- data.frame(array1, array2)
print('logically_invalid_interpretation')
print(gwet.ac1.raw(annotations)$est$coeff.val)
array1 <- c(1,1,1,1)
array2 <- c(1,1,1,0)
annotations <- data.frame(array1, array2)
print('semantic_misinterpretation')
print(gwet.ac1.raw(annotations)$est$coeff.val)
array1 <- c(1,1,1,1)
array2 <- c(1,1,1,1)
annotations <- data.frame(array1, array2)
print('suboptimal_methodology')
print(gwet.ac1.raw(annotations)$est$coeff.val)
array1 <- c(1,1,1,1)
array2 <- c(1,1,1,1)
annotations <- data.frame(array1, array2)
print('contradiction_with_established_kno

RESULTS:

[1] "Overall"

[1] 0.88308

[1] "methodology_result_conflict"

[1] 1

[1] "logically_invalid_interpretation"

[1] 0.53846

[1] "semantic_misinterpretation"

[1] 0.68

[1] "suboptimal_methodology"

[1] 1

[1] "contradiction_with_established_knowledge"

[1] 1

In [92]:
iaa_metric = 'percent_agreement'
print(f"{'category':50} {iaa_metric:5}")
print_iaa_scores(compute_iaa_score(final_annotations, iaa_metric='percent_agreement'))

category                                           percent_agreement
overall                                            0.895
methodology_result_conflict                          1.0
logically_invalid_interpretation                   0.667
semantic_misinterpretation                          0.75
suboptimal_methodology                               1.0
contradiction_with_established_knowledge             1.0


In [93]:
percentage = compute_iaa_score(final_annotations, iaa_metric='percent_agreement')

In [95]:
annotator_results = {'overall': [[] for _ in range(NUM_ANNOTATORS)]}
for meta_id, responses in final_annotations.items():
    manuscript_id, error_id, round_id = meta_id.split('___')

    for i, res in enumerate(responses):
        annotator_results['overall'][i].append(res)

        if error_id not in annotator_results:
            annotator_results[error_id] = [[] for _ in range(NUM_ANNOTATORS)]
        
        annotator_results[error_id][i].append(res)

# Pipeline Success Rate

In [104]:
annotations = annotations_1[1]

In [116]:
success_rate = {}
for annotation in annotations:
    paper_id = annotation['data']['id']
    error_id = annotation['data']['error_id']
    meta_id = f'{paper_id}___{error_id}___{round_id}'
    annotation_value = get_choice_value(annotation)
    error_name = base64.urlsafe_b64decode(error_id).decode()
    success_rate[error_name] = success_rate.get(error_name, []) + [annotation_value]

In [120]:
for error_name,results in success_rate.items():
    total = len(results)
    skipped = len([v for v in results if v is None])
    total_non_skipped = len([v for v in results if v is not None])
    successful = len([v for v in results if v == 1])
    success_percent = successful / total_non_skipped

    print(f'{error_name:50} {total:5} {skipped:5} {successful} {success_percent:5.2f}')

methodology_result_conflict                           33     3 29  0.97
logically_invalid_interpretation                      33     5 27  0.96
semantic_misinterpretation                            33     2 29  0.94
suboptimal_methodology                                33     4 13  0.45
contradiction_with_established_knowledge              33     9 23  0.96


In [121]:
for error_name,results in success_rate.items():
    total = len(results)
    skipped = len([v for v in results if v is None])
    total_non_skipped = len([v for v in results if v is not None])
    successful = len([v for v in results if v == 1])
    success_percent = successful / total

    print(f'{error_name:50} {total:5} {skipped:5} {successful} {success_percent:5.2f}')

methodology_result_conflict                           33     3 29  0.88
logically_invalid_interpretation                      33     5 27  0.82
semantic_misinterpretation                            33     2 29  0.88
suboptimal_methodology                                33     4 13  0.39
contradiction_with_established_knowledge              33     9 23  0.70


In [125]:
success_rate.values()

dict_values([[1, 1, 1, 1, 1, 1, None, 1, 1, 1, 1, 1, 1, 1, None, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, None, 1, 1, 1, 1], [None, 1, 1, 1, None, 1, 1, None, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, None, None, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, None, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, None, 1, 1, 1, 1], [1, 1, 0, 0, 1, 1, None, 0, 0, 1, 1, 0, 0, 0, 1, None, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, None, None, 0, 0], [1, 1, 1, 1, None, 0, None, 1, None, None, None, 1, 1, 1, 1, None, None, 1, None, 1, 1, 1, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

In [127]:
all_results = []
for v in success_rate.values():
    all_results = all_results + v

total = len(all_results)
skipped = len([v for v in all_results if v is None])
total_non_skipped = len([v for v in all_results if v is not None])
successful = len([v for v in all_results if v == 1])
success_percent = successful / total

print(f'{'Overall (w. skipped)':50} {total:5} {skipped:5} {successful} {success_percent:5.2f}')

success_percent = successful / total_non_skipped
print(f'{'Overall (w/o skipped)':50} {total:5} {skipped:5} {successful} {success_percent:5.2f}')

Overall (w. skipped)                                 165    23 121  0.73
Overall (w/o skipped)                                165    23 121  0.85
